In [0]:
%sql
USE CATALOG databricksdemo;
USE SCHEMA silver;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@storageaccountsunnybest.dfs.core.windows.net/products")
df = df.drop("_rescued_data")
display(df)

## Functions

In [0]:
df.createOrReplaceTempView('products');
display(spark.sql("SELECT * FROM products"))

In [0]:
%sql
CREATE OR REPLACE FUNCTION discount_func(price double) 
RETURNS double 
LANGUAGE SQL
RETURN price * 0.95

In [0]:
%sql
SELECT *,  discount_func(price)
FROM products

In [0]:
df.withColumn("discounted_price", expr("discount_func(price)")).display()

In [0]:
%sql
CREATE OR REPLACE FUNCTION upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS 
$$
return p_brand.upper()
$$

In [0]:
%sql
SELECT product_id, brand, upper_func(brand) brand_upper
FROM products;

In [0]:
df.selectExpr("product_id","brand", "upper_func(brand) AS upper_brand").display()

In [0]:
df.select("product_id","brand", expr("upper_func(brand) AS upper_brand")).display()

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("abfss://silver@storageaccountsunnybest.dfs.core.windows.net/products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS products_silver
USING delta
LOCATION 'abfss://silver@storageaccountsunnybest.dfs.core.windows.net/products'

In [0]:
%sql
SELECT *
FROM products_silver;